# LangSmith Custom Evaluation Lab

**Domain:** German language learning — sentence correction Q&A  
**Dataset:** 15 handwritten examples: a wrong German sentence + a question, with the corrected version as expected output.  
**Models:** gpt-5.4-mini at temperature=0 vs temperature=0.7 (A/B comparison).  

gpt-5.4-mini is OpenAI's current high-volume mini model: $0.75/1M input tokens, $4.50/1M output tokens, 400K context window, knowledge cutoff Aug 2025.

## Part 1: Setup

In [ ]:
%pip install langsmith openai openevals python-dotenv -q

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()  # needs LANGSMITH_API_KEY and OPENAI_API_KEY in .env

os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"]  = "german-grammar-eval"

In [2]:
from langsmith import Client, traceable
from langsmith.wrappers import wrap_openai
from openai import OpenAI
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT

client        = Client()
openai_client = wrap_openai(OpenAI())

print("Connected to LangSmith:", client.read_project(project_name="german-grammar-eval").name)

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Connected to LangSmith: german-grammar-eval


## Part 2: Dataset

15 examples. Each has:
- `sentence` — a German sentence with 1 error (or none)
- `question` — what the learner asks
- `answer` — the corrected sentence with a short explanation

Errors covered: verb-second rule, dative/accusative mix-ups, auxiliary selection (sein vs. haben), adjective endings, separable verbs, possessive inflection.

In [3]:
examples = [
    {
        "sentence": "Ich habe den Mann geholfen.",
        "question": "Is this sentence correct? If not, fix it.",
        "answer":   "Incorrect. 'helfen' takes the dative, not accusative. Correct: 'Ich habe dem Mann geholfen.'"
    },
    {
        "sentence": "Heute ich gehe in die Schule.",
        "question": "Is this sentence correct? If not, fix it.",
        "answer":   "Incorrect. In German main clauses the verb must be second. Correct: 'Heute gehe ich in die Schule.'"
    },
    {
        "sentence": "Er ruft seine Mutter an nicht.",
        "question": "Is this sentence correct? If not, fix it.",
        "answer":   "Incorrect. 'nicht' goes before the separated prefix, which itself goes to the end. Correct: 'Er ruft seine Mutter nicht an.'"
    },
    {
        "sentence": "Das ist ein gut Mann.",
        "question": "Is this sentence correct? If not, fix it.",
        "answer":   "Incorrect. After 'ein' the adjective needs a strong ending. Correct: 'Das ist ein guter Mann.'"
    },
    {
        "sentence": "Ich freue mich über das Geschenk.",
        "question": "Is this sentence correct? If not, fix it.",
        "answer":   "Correct. 'sich freuen über' uses accusative and the sentence is fine."
    },
    {
        "sentence": "Sie hat mit ihr Mutter gesprochen.",
        "question": "Is this sentence correct? If not, fix it.",
        "answer":   "Incorrect. 'mit' takes dative, so 'ihr' needs to inflect. Correct: 'Sie hat mit ihrer Mutter gesprochen.'"
    },
    {
        "sentence": "Das Buch ist interessant.",
        "question": "Is this sentence correct? If not, fix it.",
        "answer":   "Correct. Predicate adjectives don't inflect in German."
    },
    {
        "sentence": "Wir haben gestern ins Kino gegangen.",
        "question": "Is this sentence correct? If not, fix it.",
        "answer":   "Incorrect. Movement verbs like 'gehen' use 'sein' as auxiliary. Correct: 'Wir sind gestern ins Kino gegangen.'"
    },
    {
        "sentence": "Er hat einen langen Brief geschrieben.",
        "question": "Is this sentence correct? If not, fix it.",
        "answer":   "Correct. Accusative after 'schreiben', adjective ending after indefinite article is correct."
    },
    {
        "sentence": "Ich weiß nicht wo er wohnt.",
        "question": "Is this sentence correct? If not, fix it.",
        "answer":   "Technically correct in informal writing, but standard written German requires a comma before the subordinate clause: 'Ich weiß nicht, wo er wohnt.'"
    },
    {
        "sentence": "Die Kinder spielen im Garten gerne.",
        "question": "Is this sentence correct? If not, fix it.",
        "answer":   "Incorrect. 'gerne' should come before the location. More natural: 'Die Kinder spielen gerne im Garten.'"
    },
    {
        "sentence": "Ich muss mein Hausaufgaben machen.",
        "question": "Is this sentence correct? If not, fix it.",
        "answer":   "Incorrect. 'Hausaufgaben' is plural, so the possessive should be 'meine'. Correct: 'Ich muss meine Hausaufgaben machen.'"
    },
    {
        "sentence": "Er interessiert sich für die Musik.",
        "question": "Is this sentence correct? If not, fix it.",
        "answer":   "Correct, though the article 'die' before 'Musik' can be dropped in general reference. Both forms are accepted."
    },
    {
        "sentence": "Obwohl er müde war, er hat weitergearbeitet.",
        "question": "Is this sentence correct? If not, fix it.",
        "answer":   "Incorrect. After a subordinate clause the main clause verb must still be in second position. Correct: 'Obwohl er müde war, hat er weitergearbeitet.'"
    },
    {
        "sentence": "Das ist das beste Restaurant in der Stadt.",
        "question": "Is this sentence correct? If not, fix it.",
        "answer":   "Correct. Superlative form 'beste' with definite article, genitive 'der Stadt' — all fine."
    },
]

print(f"{len(examples)} examples ready")

15 examples ready


In [4]:
DATASET_NAME = "german-grammar-correction-v1"

existing = [d.name for d in client.list_datasets()]
if DATASET_NAME not in existing:
    dataset = client.create_dataset(
        DATASET_NAME,
        description="German learner sentences with 1 error each. Task: identify if correct, fix if wrong."
    )
    client.create_examples(
        inputs=[{"sentence": e["sentence"], "question": e["question"]} for e in examples],
        outputs=[{"answer": e["answer"]} for e in examples],
        dataset_id=dataset.id,
    )
    print(f"Created '{DATASET_NAME}' with {len(examples)} examples")
else:
    print(f"'{DATASET_NAME}' already exists — skipping")

'german-grammar-correction-v1' already exists — skipping


## Part 3: Target functions

Both functions use gpt-5.4-mini. A/B comparison: temperature=0 (deterministic) vs temperature=0.7 (more varied).
Model string: `gpt-5.4-mini` — $0.75/1M input, $4.50/1M output.

In [5]:
SYSTEM_PROMPT = """You are a German grammar tutor for learners at B1 level.
The user gives you a German sentence and asks whether it is correct.
If it is correct, say so briefly and explain why.
If it is wrong, identify the error, explain the rule, and provide the corrected sentence.
Keep answers under 3 sentences. Use plain English for explanations."""


@traceable(name="gpt54mini-temp0")
def target_temp0(inputs: dict) -> dict:
    """gpt-5.4-mini at temperature=0 — deterministic baseline."""
    user_msg = f"Sentence: {inputs['sentence']}\n{inputs['question']}"
    response = openai_client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        max_completion_tokens=200,
        temperature=0,
    )
    return {"answer": response.choices[0].message.content.strip()}


@traceable(name="gpt54mini-temp07")
def target_temp07(inputs: dict) -> dict:
    """gpt-5.4-mini at temperature=0.7 — more varied phrasing."""
    user_msg = f"Sentence: {inputs['sentence']}\n{inputs['question']}"
    response = openai_client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        max_completion_tokens=200,
        temperature=0.7,
    )
    return {"answer": response.choices[0].message.content.strip()}


# Quick sanity check
test_out = target_temp0(examples[0])
print("Test output (temp=0):", test_out["answer"])

Test output (temp=0): Not correct. In German, **helfen** takes the **dative** case, so it should be **dem Mann**, not **den Mann**.  
Correct sentence: **Ich habe dem Mann geholfen.**


## Part 4: Evaluators

In [6]:
# Correctness evaluator — LLM-as-judge via openevals
_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    model="anthropic:claude-haiku-4-5-20251001",
    feedback_key="correctness",
)

def correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    return _judge(inputs=inputs, outputs=outputs, reference_outputs=reference_outputs)

In [7]:
# Custom evaluator: does the answer actually include a corrected German sentence when one is needed?
# Catches cases where the model spots the error but forgets to write the fix.

CORRECTION_PROMPT = """You are evaluating a German grammar tutor's response.

Input sentence: {sentence}
Reference answer: {reference_answer}
Model answer: {model_answer}

Check: if the reference answer says the sentence is incorrect, does the model answer include a corrected German sentence?
Score 1 if the model provides a corrected sentence (or correctly says the sentence is fine).
Score 0 if the model says the sentence is wrong but fails to provide the corrected form.

Respond with JSON only: {{"score": 0 or 1, "reason": "one sentence"}}"""


def correction_completeness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    import json
    prompt = CORRECTION_PROMPT.format(
        sentence=inputs["sentence"],
        reference_answer=reference_outputs["answer"],
        model_answer=outputs["answer"],
    )
    response = openai_client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=80,
        temperature=0,
        response_format={"type": "json_object"},
    )
    result = json.loads(response.choices[0].message.content)
    return {"key": "correction_completeness", "score": result["score"], "comment": result.get("reason", "")}

## Part 5: Run evaluations

Run A (temp=0) and Run B (temp=0.7), same dataset and evaluators.

In [8]:
results_temp0 = client.evaluate(
    target_temp0,
    data=DATASET_NAME,
    evaluators=[correctness_evaluator, correction_completeness_evaluator],
    experiment_prefix="gpt-5.4-mini-temp0",
    max_concurrency=2,
)
print("Done — gpt-5.4-mini temperature=0")

View the evaluation results for experiment: 'gpt-5.4-mini-temp0-34e9e7c7' at:
https://eu.smith.langchain.com/o/30ead082-8827-4ec3-92a9-b9a1016d44ce/datasets/626f6829-6821-4799-b04c-4748a74b0918/compare?selectedSessions=c683fe08-1e32-4a05-a00a-ff0f81a62c00




15it [00:45,  3.04s/it]

Done — gpt-5.4-mini temperature=0


In [9]:
results_temp07 = client.evaluate(
    target_temp07,
    data=DATASET_NAME,
    evaluators=[correctness_evaluator, correction_completeness_evaluator],
    experiment_prefix="gpt-5.4-mini-temp07",
    max_concurrency=2,
)
print("Done — gpt-5.4-mini temperature=0.7")

View the evaluation results for experiment: 'gpt-5.4-mini-temp07-59bac9a3' at:
https://eu.smith.langchain.com/o/30ead082-8827-4ec3-92a9-b9a1016d44ce/datasets/626f6829-6821-4799-b04c-4748a74b0918/compare?selectedSessions=1162aac8-d9b0-4a28-ada7-9b092ff55202




15it [00:44,  2.96s/it]

Done — gpt-5.4-mini temperature=0.7


## Part 6: Analysis

In [10]:
import pandas as pd

def extract_scores(results, label):
    rows = []
    for r in results._results:
        fb = r.get("feedback", {})
        rows.append({
            "run":                     label,
            "correctness":             fb.get("correctness"),
            "correction_completeness": fb.get("correction_completeness"),
        })
    return pd.DataFrame(rows)

df_t0  = extract_scores(results_temp0,  "temp=0")
df_t07 = extract_scores(results_temp07, "temp=0.7")
df     = pd.concat([df_t0, df_t07], ignore_index=True)

summary = df.groupby("run")[["correctness", "correction_completeness"]].agg(["mean", "count"])
print(summary)

         correctness       correction_completeness      
                mean count                    mean count
run                                                     
temp=0           NaN     0                     NaN     0
temp=0.7         NaN     0                     NaN     0


In [11]:
# Cost estimate — gpt-5.4-mini: $0.75/1M input, $4.50/1M output
# Each example: ~120 input tokens, ~80 output tokens (rough estimate)
# Two runs = 30 total API calls for generation + ~30 calls for the custom evaluator

n = len(examples)
input_price_per_token  = 0.75  / 1_000_000
output_price_per_token = 4.50  / 1_000_000

cost_per_run = n * (120 * input_price_per_token + 80 * output_price_per_token)
cost_both    = cost_per_run * 2

print(f"Estimated cost per run (15 examples): ${cost_per_run:.5f}")
print(f"Estimated cost for both runs:         ${cost_both:.5f}")
print(f"\ngpt-5.4-mini pricing: $0.75/1M input, $4.50/1M output")

Estimated cost per run (15 examples): $0.00675
Estimated cost for both runs:         $0.01350

gpt-5.4-mini pricing: $0.75/1M input, $4.50/1M output


In [12]:
# Low-score examples for temp=0 run
low_scores = []
for i, r in enumerate(results_temp0._results):
    score = r.get("feedback", {}).get("correctness")
    if score is not None and score < 1:
        low_scores.append({
            "example_index": i,
            "sentence":      examples[i]["sentence"] if i < len(examples) else "n/a",
            "correctness":   score,
        })

print(f"temp=0 low-score examples ({len(low_scores)} total):")
for item in low_scores:
    print(f"  [{item['example_index']}] score={item['correctness']:.2f} | {item['sentence']}")

temp=0 low-score examples (0 total):
